In [ ]:
from web3 import Web3
from pathlib import Path
import json

w3 = Web3(Web3.HTTPProvider("http://127.0.0.1:8545"))
assert w3.is_connected(), "Web3 не подключён к Hardhat node"

artifact_path = (
    Path.cwd()
    / "artifacts"
    / "contracts"
    / "Token.sol"
    / "Token.json"
)


with artifact_path.open("r", encoding="utf-8") as f:
    artifact = json.load(f)

abi = artifact["abi"]
bytecode = artifact["bytecode"]

# адреса
accounts = w3.eth.accounts
sender = accounts[0]
receiver1 = accounts[1]
receiver2 = accounts[2]

initial_supply = w3.to_wei(100, "ether")
amount20 = w3.to_wei(20, "ether")

print("Accounts:")
print("Sender    :", sender)
print("Receiver_1:", receiver1)
print("Receiver_2:", receiver2)
print("-----------")

Token = w3.eth.contract(abi=abi, bytecode=bytecode)

tx_hash = Token.constructor(initial_supply).transact({"from": sender})
deploy_receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

token_address = deploy_receipt.contractAddress
print("Contract deployed at:", token_address)
print("Deploy gasUsed:", deploy_receipt.gasUsed)
print("Deploy status:", deploy_receipt.status)  # 1 = успешно

token = w3.eth.contract(address=token_address, abi=abi)

tx1_hash = token.functions.transfer(receiver1, amount20).transact(
    {"from": sender}
)
tx1_receipt = w3.eth.wait_for_transaction_receipt(tx1_hash)

print("\nTx1: Sender -> Receiver_1")
print("Tx1 gasUsed:", tx1_receipt.gasUsed)
print("Tx1 status:", tx1_receipt.status)

tx2_hash = token.functions.transfer(receiver2, amount20).transact(
    {"from": sender}
)
tx2_receipt = w3.eth.wait_for_transaction_receipt(tx2_hash)

print("\nTx2: Sender -> Receiver_2")
print("Tx2 gasUsed:", tx2_receipt.gasUsed)
print("Tx2 status:", tx2_receipt.status)

bal_sender = token.functions.balanceOf(sender).call()
bal_r1 = token.functions.balanceOf(receiver1).call()
bal_r2 = token.functions.balanceOf(receiver2).call()

print("\nFinal balances (in tokens):")
print("Sender    :", w3.from_wei(bal_sender, "ether"))
print("Receiver_1:", w3.from_wei(bal_r1, "ether"))
print("Receiver_2:", w3.from_wei(bal_r2, "ether"))


Accounts:
Sender    : 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
Receiver_1: 0x70997970C51812dc3A010C7d01b50e0d17dc79C8
Receiver_2: 0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC
-----------
Contract deployed at: 0x5FbDB2315678afecb367f032d93F642f64180aa3
Deploy gasUsed: 966045
Deploy status: 1

Tx1: Sender -> Receiver_1
Tx1 gasUsed: 52222
Tx1 status: 1

Tx2: Sender -> Receiver_2
Tx2 gasUsed: 52210
Tx2 status: 1

Final balances (in tokens):
Sender    : 60
Receiver_1: 20
Receiver_2: 20
